# Exploratory Data Analysis
## Explainable Federated Ensemble Learning for Real-Time Financial Fraud Detection

This notebook performs EDA across the three project datasets:
1. ULB Credit Card Fraud Detection (Zenodo, 2013)
2. Bank Account Fraud — NeurIPS 2022 (BAF Base)
3. Synthetic Financial Transactions (Kaggle)

Goals:
- Understand class distributions and imbalance ratios
- Profile feature types, ranges, and missing values
- Identify correlations with fraud labels
- Assess temporal patterns
- Document dataset suitability for federated learning simulation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "figure.figsize": (10, 6),
})

FIGURES_DIR = Path("../outputs/figures")
TABLES_DIR = Path("../outputs/tables")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Datasets

In [ ]:
RAW_DIR = Path("../data/raw")

# Dataset 1: ULB (284,807 transactions, 0.17% fraud)
df_ulb = pd.read_csv(RAW_DIR / "ulb_creditcard.csv")
print(f"ULB: {df_ulb.shape[0]:,} rows, {df_ulb.shape[1]} cols")

# Dataset 2: BAF Base (1,000,000 transactions, 1.10% fraud)
df_baf = pd.read_csv(RAW_DIR / "baf_base.csv")
print(f"BAF: {df_baf.shape[0]:,} rows, {df_baf.shape[1]} cols")

# Dataset 3: Synthetic (5,000,000 transactions, 3.59% fraud)
# Load a stratified sample for EDA — full set used in training later
df_synth_full = pd.read_csv(RAW_DIR / "synthetic_fraud.csv", dtype={"is_fraud": str})
df_synth_full["is_fraud"] = df_synth_full["is_fraud"].map({"True": 1, "False": 0})
print(f"Synthetic: {df_synth_full.shape[0]:,} rows, {df_synth_full.shape[1]} cols")

## 2. Dataset Summary Table

In [ ]:
def summarise_dataset(df, name, target_col):
    """Compute key statistics for a single dataset."""
    n = len(df)
    n_fraud = df[target_col].sum()
    n_legit = n - n_fraud
    fraud_rate = n_fraud / n * 100
    n_missing = df.isnull().sum().sum()
    return {
        "Dataset": name,
        "Transactions": f"{n:,}",
        "Legitimate": f"{n_legit:,}",
        "Fraudulent": f"{n_fraud:,}",
        "Fraud Rate (%)": f"{fraud_rate:.3f}",
        "Imbalance Ratio": f"{n_legit // max(n_fraud, 1)}:1",
        "Features": df.shape[1],
        "Missing Values": f"{n_missing:,}",
    }

summary = pd.DataFrame([
    summarise_dataset(df_ulb, "ULB (2013)", "Class"),
    summarise_dataset(df_baf, "BAF NeurIPS (2022)", "fraud_bool"),
    summarise_dataset(df_synth_full, "Synthetic (Kaggle)", "is_fraud"),
])

summary.to_csv(TABLES_DIR / "dataset_summary.csv", index=False)
summary

## 3. Class Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

datasets = [
    (df_ulb, "Class", "ULB (2013)"),
    (df_baf, "fraud_bool", "BAF NeurIPS (2022)"),
    (df_synth_full, "is_fraud", "Synthetic (Kaggle)"),
]

for ax, (df, col, title) in zip(axes, datasets):
    counts = df[col].value_counts().sort_index()
    bars = ax.bar(
        ["Legitimate", "Fraud"], counts.values,
        color=["#3498db", "#c0392b"], edgecolor="black", linewidth=0.5
    )
    ax.set_title(title)
    ax.set_ylabel("Count")
    for bar, val in zip(bars, counts.values):
        pct = val / len(df) * 100
        ax.text(
            bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f"{val:,}\n({pct:.2f}%)",
            ha="center", va="bottom", fontsize=8
        )
    ax.set_yscale("log")
    ax.set_ylabel("Count (log scale)")

fig.suptitle("Class Distribution Across Datasets", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "class_distribution_all.png", bbox_inches="tight")
plt.show()

---
## 4. ULB Dataset — Detailed EDA

In [ ]:
print("Shape:", df_ulb.shape)
print("\nColumn types:")
print(df_ulb.dtypes.value_counts())
print("\nBasic statistics (Amount):")
print(df_ulb.groupby("Class")["Amount"].describe().round(2))

In [ ]:
# Transaction amount distributions by class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_ulb[df_ulb["Class"] == 0]["Amount"].clip(upper=500).hist(
    bins=50, ax=axes[0], color="#3498db", alpha=0.8, edgecolor="black", linewidth=0.3
)
axes[0].set_title("Legitimate Transactions")
axes[0].set_xlabel("Amount (clipped at 500)")
axes[0].set_ylabel("Frequency")

df_ulb[df_ulb["Class"] == 1]["Amount"].hist(
    bins=50, ax=axes[1], color="#c0392b", alpha=0.8, edgecolor="black", linewidth=0.3
)
axes[1].set_title("Fraudulent Transactions")
axes[1].set_xlabel("Amount")
axes[1].set_ylabel("Frequency")

fig.suptitle("ULB — Transaction Amount Distribution", fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "ulb_amount_distribution.png", bbox_inches="tight")
plt.show()

In [ ]:
# Temporal distribution of fraud
fig, ax = plt.subplots(figsize=(12, 4))

hours = df_ulb["Time"] / 3600
ax.hist(hours[df_ulb["Class"] == 0], bins=48, alpha=0.5, color="#3498db",
        label="Legitimate", density=True)
ax.hist(hours[df_ulb["Class"] == 1], bins=48, alpha=0.7, color="#c0392b",
        label="Fraud", density=True)
ax.set_xlabel("Time (hours from first transaction)")
ax.set_ylabel("Density")
ax.set_title("ULB — Temporal Distribution of Transactions")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "ulb_temporal_distribution.png", bbox_inches="tight")
plt.show()

In [ ]:
# Correlation with fraud target
corr_with_fraud = df_ulb.corr(numeric_only=True)["Class"].drop("Class")
top_corr = corr_with_fraud.abs().sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
colours = ["#c0392b" if corr_with_fraud[f] < 0 else "#3498db" for f in top_corr.index]
ax.barh(top_corr.index[::-1], top_corr.values[::-1], color=colours[::-1],
        edgecolor="black", linewidth=0.3)
ax.set_xlabel("Absolute Correlation with Fraud")
ax.set_title("ULB — Top 15 Features Correlated with Fraud")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "ulb_top_correlated_features.png", bbox_inches="tight")
plt.show()

In [ ]:
# Full correlation heatmap
plt.figure(figsize=(16, 13))
corr_matrix = df_ulb.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, cmap="RdBu_r", center=0,
    square=True, linewidths=0.3, cbar_kws={"shrink": 0.8},
    xticklabels=True, yticklabels=True
)
plt.title("ULB — Feature Correlation Matrix", fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "ulb_correlation_heatmap.png", bbox_inches="tight")
plt.show()

---
## 5. BAF Dataset — Detailed EDA

In [ ]:
print("Shape:", df_baf.shape)
print("\nColumn types:")
print(df_baf.dtypes.value_counts())
print("\nCategorical columns:")
cat_cols = df_baf.select_dtypes(include="object").columns.tolist()
for col in cat_cols:
    print(f"  {col}: {df_baf[col].nunique()} unique -> {df_baf[col].unique().tolist()}")
print("\nNumerical summary:")
df_baf.describe().round(3)

In [ ]:
# Fraud rate by month — temporal drift
monthly = df_baf.groupby("month").agg(
    total=("fraud_bool", "count"),
    fraud=("fraud_bool", "sum")
)
monthly["fraud_rate"] = monthly["fraud"] / monthly["total"] * 100

fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.bar(monthly.index, monthly["total"], color="#3498db", alpha=0.6,
        label="Total Transactions", edgecolor="black", linewidth=0.3)
ax1.set_xlabel("Month")
ax1.set_ylabel("Transaction Count", color="#3498db")
ax1.tick_params(axis="y", labelcolor="#3498db")

ax2 = ax1.twinx()
ax2.plot(monthly.index, monthly["fraud_rate"], "o-", color="#c0392b",
         linewidth=2, markersize=6, label="Fraud Rate (%)")
ax2.set_ylabel("Fraud Rate (%)", color="#c0392b")
ax2.tick_params(axis="y", labelcolor="#c0392b")

fig.suptitle("BAF — Monthly Transaction Volume and Fraud Rate (Temporal Drift)", fontsize=13)
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "baf_monthly_fraud_drift.png", bbox_inches="tight")
plt.show()

In [ ]:
# Key numerical features: fraud vs legitimate
num_features = [
    "income", "customer_age", "credit_risk_score",
    "velocity_6h", "velocity_24h", "session_length_in_minutes",
    "proposed_credit_limit", "days_since_request"
]

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, feat in enumerate(num_features):
    ax = axes[i]
    for label, colour in [(0, "#3498db"), (1, "#c0392b")]:
        subset = df_baf[df_baf["fraud_bool"] == label][feat].dropna()
        ax.hist(subset, bins=40, alpha=0.5, color=colour, density=True,
                label="Legit" if label == 0 else "Fraud",
                edgecolor="black", linewidth=0.2)
    ax.set_title(feat, fontsize=10)
    ax.legend(fontsize=7)

fig.suptitle("BAF — Feature Distributions by Class", fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "baf_feature_distributions.png", bbox_inches="tight")
plt.show()

In [ ]:
# Fraud rate by categorical features
cat_features = ["payment_type", "employment_status", "housing_status",
                "source", "device_os"]

fig, axes = plt.subplots(1, 5, figsize=(20, 5))

for ax, feat in zip(axes, cat_features):
    fraud_by_cat = df_baf.groupby(feat)["fraud_bool"].mean() * 100
    fraud_by_cat.sort_values().plot(kind="barh", ax=ax, color="#c0392b",
                                    edgecolor="black", linewidth=0.3)
    ax.set_title(feat, fontsize=10)
    ax.set_xlabel("Fraud Rate (%)")

fig.suptitle("BAF — Fraud Rate by Categorical Features", fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "baf_fraud_rate_categorical.png", bbox_inches="tight")
plt.show()

In [ ]:
# Correlation heatmap (numerical columns only)
baf_numeric = df_baf.select_dtypes(include=[np.number])
corr_baf = baf_numeric.corr()

plt.figure(figsize=(16, 13))
mask = np.triu(np.ones_like(corr_baf, dtype=bool))
sns.heatmap(
    corr_baf, mask=mask, cmap="RdBu_r", center=0,
    square=True, linewidths=0.3, cbar_kws={"shrink": 0.8},
    xticklabels=True, yticklabels=True
)
plt.title("BAF — Feature Correlation Matrix", fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "baf_correlation_heatmap.png", bbox_inches="tight")
plt.show()

In [ ]:
# Top correlated features with fraud
corr_baf_fraud = corr_baf["fraud_bool"].drop("fraud_bool")
top_baf = corr_baf_fraud.abs().sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
colours = ["#c0392b" if corr_baf_fraud[f] < 0 else "#3498db" for f in top_baf.index]
ax.barh(top_baf.index[::-1], top_baf.values[::-1], color=colours[::-1],
        edgecolor="black", linewidth=0.3)
ax.set_xlabel("Absolute Correlation with Fraud")
ax.set_title("BAF — Top 15 Features Correlated with Fraud")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "baf_top_correlated_features.png", bbox_inches="tight")
plt.show()

---
## 6. Synthetic Dataset — Detailed EDA

In [ ]:
print("Shape:", df_synth_full.shape)
print("\nColumn types:")
print(df_synth_full.dtypes.value_counts())
print("\nCategorical columns:")
cat_cols_synth = df_synth_full.select_dtypes(include="object").columns.tolist()
for col in cat_cols_synth:
    print(f"  {col}: {df_synth_full[col].nunique()} unique")
print("\nMissing values:")
missing = df_synth_full.isnull().sum()
print(missing[missing > 0])

In [ ]:
# Transaction amount by class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_synth_full[df_synth_full["is_fraud"] == 0]["amount"].clip(upper=5000).hist(
    bins=50, ax=axes[0], color="#3498db", alpha=0.8, edgecolor="black", linewidth=0.3
)
axes[0].set_title("Legitimate Transactions")
axes[0].set_xlabel("Amount (clipped at 5000)")
axes[0].set_ylabel("Frequency")

df_synth_full[df_synth_full["is_fraud"] == 1]["amount"].clip(upper=5000).hist(
    bins=50, ax=axes[1], color="#c0392b", alpha=0.8, edgecolor="black", linewidth=0.3
)
axes[1].set_title("Fraudulent Transactions")
axes[1].set_xlabel("Amount (clipped at 5000)")
axes[1].set_ylabel("Frequency")

fig.suptitle("Synthetic — Transaction Amount Distribution", fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "synth_amount_distribution.png", bbox_inches="tight")
plt.show()

In [ ]:
# Fraud rate by categorical features
synth_cats = ["transaction_type", "merchant_category", "device_used",
              "payment_channel", "location"]

fig, axes = plt.subplots(1, 5, figsize=(22, 5))

for ax, feat in zip(axes, synth_cats):
    fraud_by_cat = df_synth_full.groupby(feat)["is_fraud"].mean() * 100
    fraud_by_cat.sort_values().plot(kind="barh", ax=ax, color="#c0392b",
                                    edgecolor="black", linewidth=0.3)
    ax.set_title(feat, fontsize=9)
    ax.set_xlabel("Fraud Rate (%)")

fig.suptitle("Synthetic — Fraud Rate by Categorical Features", fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "synth_fraud_rate_categorical.png", bbox_inches="tight")
plt.show()

In [ ]:
# Numerical feature distributions
synth_num = ["amount", "spending_deviation_score", "velocity_score",
             "geo_anomaly_score", "time_since_last_transaction"]

fig, axes = plt.subplots(1, 5, figsize=(22, 4))

for ax, feat in zip(axes, synth_num):
    for label, color in [(0, "#3498db"), (1, "#c0392b")]:
        subset = df_synth_full[df_synth_full["is_fraud"] == label][feat].dropna()
        ax.hist(subset, bins=40, alpha=0.5, color=color, density=True,
                label="Legit" if label == 0 else "Fraud",
                edgecolor="black", linewidth=0.2)
    ax.set_title(feat, fontsize=9)
    ax.legend(fontsize=7)

fig.suptitle("Synthetic — Numerical Feature Distributions by Class", fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "synth_feature_distributions.png", bbox_inches="tight")
plt.show()

---
## 7. Cross-Dataset Comparison

In [ ]:
# Side-by-side fraud rate comparison
fraud_rates = {
    "ULB (2013)": df_ulb["Class"].mean() * 100,
    "BAF (2022)": df_baf["fraud_bool"].mean() * 100,
    "Synthetic": df_synth_full["is_fraud"].mean() * 100,
}

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(fraud_rates.keys(), fraud_rates.values(),
              color=["#3498db", "#2ecc71", "#f39c12"],
              edgecolor="black", linewidth=0.5)
for bar, val in zip(bars, fraud_rates.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f"{val:.3f}%", ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.set_ylabel("Fraud Rate (%)")
ax.set_title("Fraud Rate Comparison Across Datasets")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "cross_dataset_fraud_rates.png", bbox_inches="tight")
plt.show()

In [ ]:
# Feature type comparison table
feature_comparison = pd.DataFrame({
    "Property": [
        "Feature Types",
        "Anonymised (PCA)",
        "Interpretable Features",
        "Temporal Feature",
        "Categorical Features",
        "Fraud Label",
        "DOI Available",
    ],
    "ULB (2013)": [
        "Numerical", "Yes (V1-V28)", "Amount only",
        "Time (seconds)", "None", "Class", "Yes",
    ],
    "BAF (2022)": [
        "Mixed", "No", "income, age, credit_risk_score, velocity, etc.",
        "month (0-7)", "payment_type, employment, housing, source, device_os",
        "fraud_bool", "Yes",
    ],
    "Synthetic": [
        "Mixed", "No", "amount, scores",
        "timestamp", "transaction_type, merchant, location, device, payment_channel",
        "is_fraud", "No",
    ],
})

feature_comparison.to_csv(TABLES_DIR / "feature_comparison.csv", index=False)
feature_comparison

---
## 8. Key Findings

### Class Imbalance
- All three datasets exhibit genuine class imbalance suitable for fraud detection research
- ULB is the most extreme (577:1), followed by BAF (90:1), then Synthetic (27:1)
- SMOTE will be required on training splits to prevent majority-class bias

### Feature Diversity
- ULB provides PCA-anonymised features — useful for benchmarking but limits SHAP interpretability to abstract components
- BAF provides fully interpretable features (income, age, velocity) — ideal for meaningful SHAP explanations
- Synthetic provides categorical-rich features (location, device, merchant) — requires encoding but adds diversity

### Temporal Patterns
- ULB shows time-dependent fraud clustering within a 48-hour window
- BAF shows increasing fraud rate over 8 months (temporal drift) — relevant for drift detection analysis

### Federated Learning Suitability
- The three datasets naturally represent different "institutions" with:
  - Different feature spaces (PCA vs interpretable vs categorical)
  - Different fraud rates (0.17% vs 1.10% vs 3.59%)
  - Different data volumes (285K vs 1M vs 5M)
- This heterogeneity is realistic for cross-institutional FL simulation

### Data Quality
- ULB and BAF: zero missing values
- Synthetic: missing values in time_since_last_transaction (17.9%) — will require imputation